In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
data_dir = './csvs/'

inf_curr_fwd = pd.read_csv(os.path.join(data_dir, 'inf_curriculum_forward.csv'))
inf_curr_rvs = pd.read_csv(os.path.join(data_dir, 'inf_curriculum_reverse.csv'))
inf_bench_fwd = pd.read_csv(os.path.join(data_dir, 'inf_benchmark_forward.csv'))
inf_bench_rvs = pd.read_csv(os.path.join(data_dir, 'inf_benchmark_reverse.csv'))

prob_curr_fwd = pd.read_csv(os.path.join(data_dir, 'probing_curriculum_forward.csv'))
prob_curr_rvs = pd.read_csv(os.path.join(data_dir, 'probing_curriculum_reverse.csv'))
prob_bench_fwd = pd.read_csv(os.path.join(data_dir, 'probing_benchmark_forward.csv'))
prob_bench_rvs = pd.read_csv(os.path.join(data_dir, 'probing_benchmark_reverse.csv'))

In [3]:
print('-'*20 + 'Inference Curriculum Forward' + '-'*20)
print(inf_curr_fwd.head())

print('-'*20 + 'Inference Curriculum Reverse' + '-'*20)
print(inf_curr_rvs.head())

print('-'*20 + 'Inference Benchmark Forward' + '-'*20)
print(inf_bench_fwd.head())

print('-'*20 + 'Inference Benchmark Reverse' + '-'*20)
print(inf_bench_rvs.head())

--------------------Inference Curriculum Forward--------------------
   model           task    metric    Base  S0_mid  S0_end  S1_mid  S1_end  \
0  llava      S0_logamr  accuracy  0.1385  0.3201  0.3208  0.3099  0.3088   
1  llava  S1_similarity       mae  0.3374  0.3415  0.3415  0.1438  0.1409   
2  llava    S2_ishihara  accuracy  0.4214  0.4357  0.4357  0.4214  0.4214   
3  llava      S3_naming  accuracy  0.6152  0.5667  0.5667  0.5788  0.5788   
4  llava       S4_depth  accuracy  0.2205  0.2227  0.2249  0.2227  0.2227   

   S2_mid  S2_end  S3_mid  S3_end  S4_mid  S4_end  
0  0.2988  0.2982  0.2812  0.2811  0.2289  0.2285  
1  0.1472  0.1473  0.1426  0.1424  0.1550  0.1583  
2  0.4357  0.4357  0.4214  0.4214  0.4214  0.4214  
3  0.6182  0.6242  0.8394  0.8394  0.8576  0.8576  
4  0.2205  0.2205  0.2160  0.2160  0.2851  0.2873  
--------------------Inference Curriculum Reverse--------------------
   model           task    metric    Base  S4_mid  S4_end  S3_mid  S3_end  \
0  llava  

In [4]:
print('-'*20 + 'Probing Curriculum Forward' + '-'*20)
print(prob_curr_fwd.head())

print('-'*20 + 'Probing Curriculum Reverse' + '-'*20)
print(prob_curr_rvs.head())

print('-'*20 + 'Probing Benchmark Forward' + '-'*20)
print(prob_bench_fwd.head())

print('-'*20 + 'Probing Benchmark Reverse' + '-'*20)
print(prob_bench_rvs.head())

--------------------Probing Curriculum Forward--------------------
   model       task     layer metric    Base  S0_mid  S0_end  S1_mid  S1_end  \
0  llava  S0_logamr   layer_0    acc  0.0667  0.0667  0.0667  0.0667  0.0667   
1  llava  S0_logamr   layer_8    acc  0.8557  0.8578  0.8583  0.8567  0.8572   
2  llava  S0_logamr  layer_16    acc  0.8933  0.8913  0.8919  0.8907  0.8917   
3  llava  S0_logamr  layer_24    acc  0.9022  0.8998  0.9022  0.9020  0.9011   
4  llava  S0_logamr  layer_32    acc  0.9019  0.9002  0.9013  0.9009  0.9015   

   S2_mid  S2_end  S3_mid  S3_end  S4_mid  S4_end  
0  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  
1  0.8580  0.8602  0.8581  0.8585  0.8530  0.8554  
2  0.8907  0.8919  0.8883  0.8887  0.8893  0.8885  
3  0.9024  0.9026  0.9004  0.9015  0.9020  0.9006  
4  0.9000  0.9006  0.9006  0.8978  0.8994  0.9002  
--------------------Probing Curriculum Reverse--------------------
   model       task     layer metric    Base  S4_mid  S4_end  S3_mid  S3_

# 귀인 분석 — 학습 결과를 가르는 것은 무엇인가

모델 / 태스크 순서(direction) / 초기 성능 중 무엇이 학습 후 변화를 설명하는가.
응답변수는 **delta = acc - base** (학습 전 대비 변화량, 정확도 단위).
각 분석은 fixed / random 을 밝히고 `lmer` + `summary()` 만 낸다.

- 변수 정의, 모형 선택 근거, 진단: **[analysis_notes.md](analysis_notes.md)**
- 방향 효과 탐색 절차: **[exploratory_direction.md](exploratory_direction.md)**

In [5]:
%load_ext rpy2.ipython

In [6]:
import numpy as np, pandas as pd

def to_long(df, direction, extra=()):
    """wide(S*_end) -> long. mid 체크포인트는 중복이라 버린다. mae 는 1-MAE 로 뒤집음."""
    stages = [4, 3, 2, 1, 0] if direction == 'reverse' else [0, 1, 2, 3, 4]
    rec = []
    for _, r in df.iterrows():
        flip = (r['metric'] == 'mae'); base = 1 - r['Base'] if flip else r['Base']
        for pos, k in enumerate(stages):
            v = r.get(f'S{k}_end')
            if pd.isna(v): continue          # 평가 실패(NaN) 제외
            row = dict(model=r['model'], task=r['task'], direction=direction,
                       time=pos + 1, base=base, acc=(1 - v) if flip else v)
            row.update({c: r[c] for c in extra}); rec.append(row)
    return pd.DataFrame(rec)

def add_base(d):
    """초기성능 between/within 분해 + delta + time_c."""
    d = d.join(d.groupby('model')['base'].mean().rename('base_bw'), on='model')
    d['base_wi'] = d['base'] - d['base_bw']
    d['delta'] = d['acc'] - d['base']; d['time_c'] = d['time'] - 3.0
    return d

def add_history(d):
    """학습 이력(trained_yet, recency). 태스크 이름의 stage 를 쓰므로 커리큘럼 전용."""
    stage = d['task'].str[1].astype(int)
    own_end = np.where(d['direction'] == 'forward', stage + 1, (4 - stage) + 1)
    d['trained_yet'] = (d['time'] >= own_end).astype(int)
    d['recency'] = np.clip(d['time'] - own_end, 0, None)
    return d

def depth_label(df):
    """layer_N -> 상대 깊이(모델별 정규화), proj 는 그대로."""
    n = df['layer'].str.extract(r'layer_(\d+)')[0].astype(float)
    L = n.groupby(df['model']).transform('max')
    return np.where(df['layer'] == 'proj', 'proj', (n / L).round(2).astype(str))

def add_depth_num(d):
    """proj -> is_proj=1 (별도 지시변수), 나머지는 상대 깊이 0~1 연속(depth_num)."""
    d['is_proj'] = (d['depth'] == 'proj').astype(int)
    d['depth_num'] = np.where(d['is_proj'] == 1, 0.0,
                              pd.to_numeric(d['depth'].where(d['depth'] != 'proj'), errors='coerce'))
    return d

DIR_CONTRAST = {'forward': 0.5, 'reverse': -0.5}

def dir_code(s):
    """direction 을 ±0.5 대비코딩. 이미 숫자면 그대로 둔다(셀 재실행 안전)."""
    return s if pd.api.types.is_numeric_dtype(s) else s.map(DIR_CONTRAST).astype(float)


## 1. inference 커리큘럼

```
fixed :  base_wi + base_bw + trained_yet + recency + direction * time_c
random:  (1|model) + (1|task) + (1|model:task)
응답  :  delta = acc - base          (245행)
```
계수는 정확도 단위. 요인별 R^2 기여, 「순서」 두 정의 해석은 [analysis_notes.md](analysis_notes.md).

In [7]:
# inference 커리큘럼: 5모델 x 5태스크 x 2방향 x 5시점 = 245행 (mid 제외)
d = pd.concat([to_long(inf_curr_fwd, 'forward'),
               to_long(inf_curr_rvs, 'reverse')], ignore_index=True)
d = add_history(add_base(d))
print('rows=%d  models=%d  tasks=%d' % (len(d), d.model.nunique(), d.task.nunique()))

rows=245  models=5  tasks=5


## 변수 설명

- `base_wi` : 한 태스크 내에서 초기 성능의 좋고 나쁨 (각 태스크에서 모델별 초기 성능 - 모델별 평균)
- `base_bw` : 모델별 전체 태스크에서의 초기 성능 평균 (모든 태스크에서의 모델별 초기 성능의 좋고 나쁨)
- `trained_yet` : 학습 전후 구분 (0=학습 전, 1=학습 후)
- `recency` : 학습 시점 (1~5)
- `direction` : 학습 순서 **대비코딩** (forward = +0.5, reverse = -0.5).
  계수는 곧 forward - reverse 차이이고, `time_c` 계수는 두 방향의 **평균** 기울기가 된다.
  (0/1 더미코딩이면 `time_c` 가 기준집단 한쪽의 기울기라서 방향을 뒤집어 두 번 적합해야 했다.)
- `time_c` : 학습 시점 중심화 (1~5 → -2~2)

In [8]:
# direction: forward = +0.5, reverse = -0.5 (sum/대비 코딩)
# -> direction 계수 = forward - reverse 차이, time_c 계수 = 두 방향 평균 기울기.
#    direction:time_c 의 추정치/p 값은 코딩과 무관하게 동일하다.
d['direction'] = dir_code(d['direction'])
d['direction'].value_counts()

direction
 0.5    125
-0.5    120
Name: count, dtype: int64

### 적합  (direction: forward +0.5 / reverse -0.5)

In [9]:
import os
import re

import numpy as np
import matplotlib.pyplot as plt

FIGDIR = "./figures"          # 그림 저장 폴더 (없으면 자동 생성)
FORMATS = ("pdf", "png")      # 저장 포맷
DPI = 300

# ── 항 이름 → (그림에 찍을 기호, caption 에 쓸 설명) ──────────────────────────
# 그림에는 기호만 넣고, 무슨 뜻인지는 caption 으로 뺀다.
# matplotlib mathtext: $...$ 안은 LaTeX 없이 렌더링.
# (R 에서 factor 계수는 뒤에 수준이 붙는다: direction → 'direction1')
TERMS = {
    "base_wi":     (r"$T_{w}$", r"within-task baseline: a model's baseline accuracy on that task "
                                r"minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$)"),
    "base_bw":     (r"$T_{b}$", r"between-model baseline: a model's mean baseline accuracy "
                                r"minus the grand mean ($\bar{T}_{m}-\bar{T}$)"),
    "trained_yet": (r"$S$",     "trained-session indicator (1 after the task has been trained)"),
    "recency":     ("$R$",      "recency of the training session for that task"),
    "direction":   ("$D$",      "curriculum direction contrast "
                                "(+0.5 = forward, -0.5 = reverse)"),
    "direction1":  ("$D$",      "curriculum direction contrast "
                                "(+0.5 = forward, -0.5 = reverse)"),   # factor 로 넣었을 때 이름
    "time_c":      ("$t$",      "centered training time (session index)"),
    "depth_num":   ("$d$",      "relative representation depth (0 = first layer, 1 = last layer)"),
    "is_proj":     ("$P$",      "projector-layer indicator"),
}

LABELS = {k: v[0] for k, v in TERMS.items()}
DESCRIPTIONS = {k: v[1] for k, v in TERMS.items()}

def _label(term, labels):
    if term in labels:
        return labels[term]
    if ":" in term:                                   # 상호작용: 각 항을 매핑해 a × b
        return r" $\times$ ".join(_label(p, labels) for p in term.split(":"))
    return term                                       # 매핑 없으면 원래 이름

def _pfmt(p):
    return "p<.001" if p < 1e-3 else ("p=%.3f" % p)

def _slug(s):
    """제목 → 파일명. 'probing · forward · direction×depth' → 'probing_forward_directionxdepth'"""
    s = s.replace("×", "x")
    s = re.sub(r"[^0-9A-Za-z]+", "_", s)
    return s.strip("_").lower() or "figure"

def _fixef(model_name):
    """R 세션의 lmer 모델에서 fixed effect 표를 꺼낸다 -> (est, se, pval, terms)"""
    from IPython import get_ipython
    import numpy as np

    ip = get_ipython()
    coef  = np.asarray(ip.run_line_magic('R', 'summary(%s)$coefficients' % model_name), dtype=float)
    terms = np.asarray(list(ip.run_line_magic('R', 'rownames(summary(%s)$coefficients)' % model_name)),
                       dtype=object)
    pval = coef[:, 4] if coef.shape[1] > 4 else None
    return coef[:, 0], coef[:, 1], pval, terms

def caption_for(terms, labels=LABELS, descs=DESCRIPTIONS):
    """그림에 실제로 들어간 항만 골라서 '기호 = 설명' caption 문장을 만든다."""
    seen, parts = set(), []
    for term in terms:                                # 상호작용은 구성 항으로 쪼개서 한 번씩만
        for part in term.split(":"):
            if part in seen or part not in descs:
                continue
            seen.add(part)
            parts.append("%s: %s" % (labels[part], descs[part]))
    return ("Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. "
            + "; ".join(parts) + ".")

def plot_fixef(model_name, title="", drop_intercept=True, labels=LABELS,
               show_value=True, show_p=True,
               save=None, outdir=FIGDIR, formats=FORMATS, dpi=DPI,
               print_caption=True):
    """R 세션의 lmer 모델에서 fixed effect 계수를 받아와 95% CI coefficient plot 을 그린다.

    save : None/False 면 저장 안 함. 문자열이면 그 이름(확장자 없이)으로,
           True 면 title 을 슬러그로 바꿔서 outdir 아래 formats 확장자마다 저장한다.
    print_caption : 그림에 들어간 기호의 설명을 caption 문장으로 출력한다.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    est, se, pval, terms = _fixef(model_name)

    if drop_intercept:
        keep = terms != "(Intercept)"
        est, se, terms = est[keep], se[keep], terms[keep]
        if pval is not None:
            pval = pval[keep]

    disp = [_label(t, labels) for t in terms]
    lo, hi = est - 1.96 * se, est + 1.96 * se
    y = np.arange(len(est))[::-1]
    sig = (lo > 0) | (hi < 0)
    colors = np.where(sig, "firebrick", "0.45")

    # [수정된 부분 1] 서로 간격을 좁게 하기 위해 y축 항목당 높이를 0.5 -> 0.35로 줄임
    fig, ax = plt.subplots(figsize=(7, 0.35 * len(est) + 1.0))

    ax.axvline(0, ls="--", color="grey", lw=1)
    ax.hlines(y, lo, hi, color=colors, lw=2)

    # [수정된 부분] 폰트가 커지면 점도 살짝 커져야 균형이 맞으므로 s=45 -> 60으로 확대
    ax.scatter(est, y, color=colors, s=60, zorder=3)

    if show_value or (show_p and pval is not None):
        span = (hi.max() - lo.min()) or 1.0
        for i in range(len(est)):
            parts = []
            if show_value:
                parts.append("%+.3f" % est[i])
            if show_p and pval is not None:
                parts.append("(%s)" % _pfmt(pval[i]))

            # [수정된 부분 2] 우측 수치 폰트 크기 키움 (fontsize 8 -> 12)
            ax.text(hi[i] + span * 0.02, y[i], " ".join(parts),
                    va="center", ha="left", fontsize=12, color=colors[i])
        ax.margins(x=0.25)

    ax.set_yticks(y)

    # [수정된 부분 3] 기호는 짧으니 조금 더 크게
    ax.set_yticklabels(disp, fontsize=16)

    # [수정된 부분 4] X축 라벨 및 눈금 폰트 크기 키움
    ax.set_xlabel("Effect size (95% CI)", fontsize=13)
    ax.tick_params(axis='x', labelsize=12)

    # 테두리를 지웠으므로 Y축의 자잘한 눈금선(tick mark)도 깔끔하게 숨김 처리
    ax.tick_params(axis='y', length=0)

    # [수정된 부분 5] 타이틀 폰트 크기 키움 (title="" 이면 제목 없는 그림)
    ax.set_title(title, loc="left", fontsize=15)

    # [수정된 부분 6] 상하좌우 바운더리(Spines) 완전 삭제
    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()

    # [추가] 파일로 저장 — plt.show() 보다 반드시 먼저.
    if save:
        stem = _slug(title or model_name) if save is True else str(save)
        os.makedirs(outdir, exist_ok=True)
        for ext in formats:
            path = os.path.join(outdir, "%s.%s" % (stem, ext))
            fig.savefig(path, dpi=dpi, bbox_inches="tight")
            print("saved:", path)

    plt.show()

    # [추가] 논문 caption 에 그대로 붙일 기호 설명
    if print_caption:
        print("\ncaption:", caption_for(terms, labels))


In [10]:
%%R -i d
library(lmerTest)
d$model <- factor(d$model)
d$task <- factor(d$task)
# direction 은 ±0.5 로 대비코딩된 수치형이라 factor 로 바꾸지 않는다.

m <- lmer(delta ~ base_wi + base_bw + trained_yet + recency + direction * time_c
            + (1 | model) + (1 | task) + (1 | model:task), data = d)
summary(m)


Linear mixed model fit by REML. t-tests use Satterthwaite's method

 [

lmerModLmerTest

]

Formula:

...

 -0.018

  0.010

 -0.022

 -0.021

필요한 패키지를 로딩중입니다: lme4
필요한 패키지를 로딩중입니다: Matrix

다음의 패키지를 부착합니다: ‘lmerTest’

The following object is masked from ‘package:lme4’:

    lmer

The following object is masked from ‘package:stats’:

    step



In [11]:
plot_fixef("m", "inference · curriculum",
            save="fixef_curriculum_inference")

saved: ./figures/fixef_curriculum_inference.pdf
saved: ./figures/fixef_curriculum_inference.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $S$: trained-session indicator (1 after the task has been trained); $R$: recency of the training session for that task; $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index).


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
plot_fixef("m", "",
            save="fixef_curriculum_inference_no_title")

saved: ./figures/fixef_curriculum_inference_no_title.pdf
saved: ./figures/fixef_curriculum_inference_no_title.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $S$: trained-session indicator (1 after the task has been trained); $R$: recency of the training session for that task; $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index).


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. probing 커리큘럼

표상 깊이를 **연속**으로 넣는다: `depth_num`(0~1, 모델별 상대 깊이) + `is_proj`(proj 별도 지시변수).
2차항은 유의하지 않아 직선 채택 (AIC 로 범주보다 우수).
```
fixed :  base_wi + base_bw + trained_yet + recency + direction * time_c + depth_num + is_proj
random:  (1|model) + (1|task) + (1|model:task) + (1|cell)
응답  :  delta                        (1470행)
```
`depth_num` 기울기 = 한 층 깊어질 때 delta 증가량. 근거는 [analysis_notes.md](analysis_notes.md).

In [13]:
# probing 커리큘럼: layer 6개 축 추가 -> 1470행
dp = pd.concat([to_long(prob_curr_fwd, 'forward', extra=('layer',)),
                to_long(prob_curr_rvs, 'reverse', extra=('layer',))], ignore_index=True)
dp['depth'] = depth_label(dp)
dp = add_history(add_base(dp))
dp = add_depth_num(dp)
dp['cell'] = dp.model + ':' + dp.task + ':' + dp.direction + ':' + dp.time.astype(str)
print('rows=%d  depths=%d  cells=%d' % (len(dp), dp.depth.nunique(), dp.cell.nunique()))

rows=1500  depths=6  cells=250


In [14]:
dp['direction'] = dir_code(dp['direction'])

### 적합  (direction: forward +0.5 / reverse -0.5)

In [15]:
%%R -i dp
library(lmerTest)
dp$model <- factor(dp$model)
dp$task <- factor(dp$task)
dp$cell <- factor(dp$cell)
# direction 은 ±0.5 대비코딩된 수치형 (factor 로 바꾸지 않는다)
# depth 는 연속(depth_num, 0~1) + proj 별도 지시변수(is_proj)
# 2차항은 유의하지 않아(직선 채택) depth_num 기울기 하나로 본다

mp <- lmer(delta ~ base_wi + base_bw + trained_yet + recency + direction * time_c
             + depth_num + is_proj
             + (1 | model) + (1 | task) + (1 | model:task) + (1 | cell), data = dp)
summary(mp)

Linear mixed model fit by REML. t-tests use Satterthwaite's method

 [

lmerModLmerTest

]

Formula:

...

  0.000

  0.000

optimizer (nloptwrap) convergence code: 0 (OK)

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')


In [16]:
plot_fixef("mp", "probing · curriculum",
            save="fixef_curriculum_probing")

saved: ./figures/fixef_curriculum_probing.pdf


saved: ./figures/fixef_curriculum_probing.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $S$: trained-session indicator (1 after the task has been trained); $R$: recency of the training session for that task; $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index); $d$: relative representation depth (0 = first layer, 1 = last layer); $P$: projector-layer indicator.


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
plot_fixef("mp", "",
            save="fixef_curriculum_probing_no_title")

saved: ./figures/fixef_curriculum_probing_no_title.pdf
saved: ./figures/fixef_curriculum_probing_no_title.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $S$: trained-session indicator (1 after the task has been trained); $R$: recency of the training session for that task; $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index); $d$: relative representation depth (0 = first layer, 1 = last layer); $P$: projector-layer indicator.


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### 방향 x 깊이  (탐색적)


In [18]:
%%R
# depth_num 을 연속으로: direction:depth_num 계수 하나가 "방향에 따라 깊이 기울기가 다른가"
mp_dir <- lmer(delta ~ base_wi + base_bw + trained_yet + recency + direction * time_c
                 + direction * depth_num + is_proj
                 + (1 | model) + (1 | task) + (1 | model:task) + (1 | cell), data = dp)
summary(mp_dir)

Linear mixed model fit by REML. t-tests use Satterthwaite's method

 [

lmerModLmerTest

]

Formula:

...


drctn:dpth_

  0.000  

optimizer (nloptwrap) convergence code: 0 (OK)

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')


In [19]:
plot_fixef("mp_dir", "probing · curriculum · direction×depth",
            save="fixef_curriculum_probing_depth")

saved: ./figures/fixef_curriculum_probing_depth.pdf
saved: ./figures/fixef_curriculum_probing_depth.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $S$: trained-session indicator (1 after the task has been trained); $R$: recency of the training session for that task; $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index); $d$: relative representation depth (0 = first layer, 1 = last layer); $P$: projector-layer indicator.


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
plot_fixef("mp_dir", "",
            save="fixef_curriculum_probing_depth_no_title")

saved: ./figures/fixef_curriculum_probing_depth_no_title.pdf
saved: ./figures/fixef_curriculum_probing_depth_no_title.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $S$: trained-session indicator (1 after the task has been trained); $R$: recency of the training session for that task; $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index); $d$: relative representation depth (0 = first layer, 1 = last layer); $P$: projector-layer indicator.


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. benchmark  (전이)

benchmark 10개는 학습 대상이 아니다 -> `trained_yet`/`recency` 없이 `direction * time_c` 만.
`direction * time_c` 가 주 검정 (방향에 따라 전이 속도가 다른가).

In [21]:
# benchmark: 학습 대상이 아니므로 trained_yet/recency 없음. 학습량은 time_c 하나로 본다.
# (1) inference benchmark  -> db
db = add_base(pd.concat([to_long(inf_bench_fwd, 'forward'),
                         to_long(inf_bench_rvs, 'reverse')], ignore_index=True))
# (2) probing benchmark    -> dbp  (layer 깊이 축 추가)
dbp = pd.concat([to_long(prob_bench_fwd, 'forward', extra=('layer',)),
                 to_long(prob_bench_rvs, 'reverse', extra=('layer',))], ignore_index=True)
dbp['depth'] = depth_label(dbp)
dbp = add_base(dbp)
dbp = add_depth_num(dbp)
dbp['cell'] = dbp.model + ':' + dbp.task + ':' + dbp.direction + ':' + dbp.time.astype(str)

# direction 대비코딩 (섹션 1·2 와 동일: forward +0.5 / reverse -0.5)
db['direction']  = dir_code(db['direction'])
dbp['direction'] = dir_code(dbp['direction'])

print('inference bench rows=%d (결측 %d 제외)  |  probing bench rows=%d'
      % (len(db), 5*10*2*5 - len(db), len(dbp)))

inference bench rows=469 (결측 31 제외)  |  probing bench rows=3000


### inference benchmark

In [22]:
%%R -i db
library(lmerTest)
db$model <- factor(db$model)
db$task <- factor(db$task)
# direction 은 ±0.5 대비코딩된 수치형 (forward +0.5 / reverse -0.5)

# 주 검정: direction:time_c  (방향에 따라 전이 속도가 다른가)
mb <- lmer(delta ~ base_wi + base_bw + direction * time_c
             + (1 | model) + (1 | task) + (1 | model:task), data = db)
summary(mb)

Linear mixed model fit by REML. t-tests use Satterthwaite's method

 [

lmerModLmerTest

]

Formula:

...

 -0.009

 -0.010

optimizer (nloptwrap) convergence code: 0 (OK)

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')


In [23]:
plot_fixef("mb", "inference · benchmark",
            save="fixef_benchmark_inference")

saved: ./figures/fixef_benchmark_inference.pdf


saved: ./figures/fixef_benchmark_inference.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index).


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
plot_fixef("mb", "",
            save="fixef_benchmark_inference_no_title")

saved: ./figures/fixef_benchmark_inference_no_title.pdf


saved: ./figures/fixef_benchmark_inference_no_title.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index).


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### probing benchmark

In [25]:
%%R -i dbp
library(lmerTest)
dbp$model <- factor(dbp$model)
dbp$task <- factor(dbp$task)
dbp$cell <- factor(dbp$cell)
# direction 은 ±0.5 대비코딩된 수치형 (forward +0.5 / reverse -0.5)
# depth 는 연속(depth_num, 0~1) + proj 별도 지시변수(is_proj)

mq <- lmer(delta ~ base_wi + base_bw + direction * time_c + depth_num + is_proj
             + (1 | model) + (1 | task) + (1 | model:task) + (1 | cell), data = dbp)
summary(mq)

Linear mixed model fit by REML. t-tests use Satterthwaite's method

 [

lmerModLmerTest

]

Formula:

...

  0.000

  0.000

optimizer (nloptwrap) convergence code: 0 (OK)

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')


In [26]:
plot_fixef("mq", "probing · benchmark",
            save="fixef_benchmark_probing")

saved: ./figures/fixef_benchmark_probing.pdf


saved: ./figures/fixef_benchmark_probing.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index); $d$: relative representation depth (0 = first layer, 1 = last layer); $P$: projector-layer indicator.


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
plot_fixef("mq", "",
            save="fixef_benchmark_probing_no_title")

saved: ./figures/fixef_benchmark_probing_no_title.pdf
saved: ./figures/fixef_benchmark_probing_no_title.png

caption: Fixed-effect estimates with 95% CIs; terms in red have a CI excluding zero. $T_{w}$: within-task baseline: a model's baseline accuracy on that task minus its own mean over tasks ($\bar{T}_{m,t}-\bar{T}_{m}$); $T_{b}$: between-model baseline: a model's mean baseline accuracy minus the grand mean ($\bar{T}_{m}-\bar{T}$); $D$: curriculum direction contrast (+0.5 = forward, -0.5 = reverse); $t$: centered training time (session index); $d$: relative representation depth (0 = first layer, 1 = last layer); $P$: projector-layer indicator.


/var/folders/8q/yjgzft41043gt_l8bqcfnzn40000gn/T/ipykernel_7480/245006743.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
